# RAG Evaluation ( A Comprehensive Analysis )

Overview

A typical RAG evaluation workflow consists of three main steps:

- Creating a dataset with questions and their expected answers
- Running your RAG application on those questions
- Using evaluators to measure how well your application performed, looking at factors
 
 like:

- Answer relevance
- Answer accuracy
- Retrieval quality




![alt text](<Screenshot 2026-03-03 154845.png>)






In [ ]:
# install dependencies

import os
from dotenv import load_dotenv
load_dotenv()

os.environ["USER_AGENT"] = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36" #for web scraping with langchain
os.environ["GROQ_API_KEY"] = "put your API key here" #for using groq's LLMs with langchain
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")    
os.environ["LANGSMITH_TRACING"]= "true"


### 1- Creating Dataset 

In [2]:
examples = [
    {
        "inputs": {"question": "What is LangChain?"},
        "outputs": {
            "answer": "LangChain is an open-source framework for building applications powered by large language models, providing components for chaining prompts, integrating tools, memory, and external data sources."
        },
    },
    {
        "inputs": {"question": "What is LangGraph?"},
        "outputs": {
            "answer": "LangGraph is a library built on top of LangChain that enables stateful, graph-based orchestration of LLM workflows, particularly useful for building multi-agent and iterative reasoning systems."
        },
    },
    {
        "inputs": {"question": "What is LangSmith?"},
        "outputs": {
            "answer": "LangSmith is a platform for debugging, monitoring, and evaluating LLM applications, offering tracing, dataset management, and performance analysis tools."
        },
    },
    {
        "inputs": {"question": "What is OpenAI?"},
        "outputs": {
            "answer": "OpenAI is an AI research and deployment company that develops large language models such as GPT, along with APIs and tools for building AI-powered applications."
        },
    },
    {
        "inputs": {"question": "What is Google?"},
        "outputs": {
            "answer": "Google is a multinational technology company known for its search engine, cloud computing services, AI research, and products like Android and YouTube."
        },
    },
    {
        "inputs": {"question": "What is Mistral AI?"},
        "outputs": {
            "answer": "Mistral AI is a European AI company that develops open-weight large language models optimized for efficiency and strong performance."
        },
    },
    {
        "inputs": {"question": "What is ChromaDB?"},
        "outputs": {
            "answer": "ChromaDB is an open-source vector database designed for storing and retrieving embeddings, commonly used in retrieval-augmented generation (RAG) systems."
        },
    },
    {
        "inputs": {"question": "What is Groq?"},
        "outputs": {
            "answer": "Groq is a hardware and AI inference company that provides high-speed LLM inference using its custom tensor streaming processor architecture."
        },
    },
    {
        "inputs": {"question": "What is Retrieval-Augmented Generation (RAG)?"},
        "outputs": {
            "answer": "Retrieval-Augmented Generation is a technique that enhances LLM responses by retrieving relevant external documents and incorporating them into the model's context before generating an answer."
        },
    },
    {
        "inputs": {"question": "What is an embedding in machine learning?"},
        "outputs": {
            "answer": "An embedding is a dense numerical vector representation of data such as text, images, or audio, capturing semantic meaning in a continuous vector space."
        },
    }
]

In [3]:
# creating a dataset and adding examples to LangSmith for evaluation

from langsmith import Client    
client = Client()

dataset_name = "RAG Evaluation Dataset"
dataset = client.create_dataset(dataset_name)

client.create_examples(
    dataset_id=dataset.id,
    examples=examples
)

{'example_ids': ['c367954d-3d4f-4ce0-8016-9535fd3212fb',
  '86261571-010f-4109-a027-d6260fd37c07',
  '4682154e-4bb6-41ef-a484-acbd8cad70d0',
  'c1ab8f36-8620-4e23-b06e-a60929d5d945',
  'b5edd82d-75c4-4ba5-8350-fee9c9bceb72',
  'cefc6a03-6607-4537-989c-e79348f359f4',
  '95e75026-eeb5-475c-88e9-19dacab52840',
  '425c46c5-bf37-4ad8-8c5b-ac59211d7a8c',
  '3418a0b4-1b06-4723-9b56-9c52827a0b1c',
  'a6cdc231-7532-4bc9-b7a2-6ab566b5a2f7'],
 'count': 10,
 'as_of': '2026-03-02T20:51:52.708862936Z'}

### 2- Defining Metrics for Dataset (using LLM as a Judge)

In [23]:
from groq import Groq

# Initialize Groq client
groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

eval_instructions = (
    "You are an expert professor specialized in grading students' answers to questions. "
    "Respond strictly with either CORRECT or INCORRECT."
)





# This function (correctness) grades the correctness of model responses by constructing a prompt for the GROQ model, which evaluates the predicted answer against the reference answer and returns a boolean indicating whether the response is correct or not.

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    user_content = f"""
You are grading the following question:
{inputs['question']}

Here is the real answer:
{reference_outputs['answer']}

You are grading the following predicted answer:
{outputs['response']}

Respond with CORRECT or INCORRECT.
Grade:
"""

    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        temperature=0,
        messages=[
            {"role": "system", "content": eval_instructions},
            {"role": "user", "content": user_content},
        ],
    )

    grade = response.choices[0].message.content.strip()

    return grade == "CORRECT"

In [24]:
## Concisions- checks whether the actual output is less than 2x the length of the expected result.

def concision(outputs: dict, reference_outputs: dict) -> bool:
    return int(len(outputs["response"]) < 2 * len(reference_outputs["answer"]))

### 3-Run Evaluations

In [25]:
# Example of calling the llama-3.3-70b-versatile model from GROQ with custom instructions
default_instructions = "Respond to the user's question in one short sentence."

def my_app(question: str, model: str = "llama-3.3-70b-versatile", instructions: str = default_instructions) -> str:
    response = groq_client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": question},
        ],
    )
    return response.choices[0].message.content

In [26]:
# This function (ls_target) serves as the target function for evaluation, taking a question as input and returning the model's response in a dictionary format that can be used for grading and analysis.
def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"])}

In [27]:
# Running the evaluation using GROQ to grade {correctness + concision} of model responses based on the provided dataset and target function.
# storing results in langsmith under the name "llama-3.3-70b-versatile_RAG_Evaluation"
experiment_results = client.evaluate(
    ls_target,  
    data=dataset_name,
    evaluators=[correctness, concision],
    experiment_prefix="llama-3.3-70b-versatile_RAG_Evaluation"
)

print("Evaluation complete.")

View the evaluation results for experiment: 'llama-3.3-70b-versatile_RAG_Evaluation-926f8cec' at:
https://smith.langchain.com/o/eaf954a2-4e9e-4932-9036-54070559faa0/datasets/e2185fa7-4bc2-4efa-8407-3b8923855912/compare?selectedSessions=91e92233-e628-480d-9c64-602c9ffee79b




10it [00:05,  1.92it/s]

Evaluation complete.


# Creating A RAG pipeline for evaluation

In [42]:

# RAG Components
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# List of URLs
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

# Load documents
docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [doc for sublist in docs for doc in sublist]

# Split documents
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

doc_splits = text_splitter.split_documents(docs_list)

In [43]:
# initialising sentence-transformer for embeddings
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 435.00it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [44]:
# Create vector store
vectorstore = InMemoryVectorStore.from_documents(
    documents=doc_splits,
    embedding=embedding_model,
)

# Create retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})

In [46]:
#checking if retriever works
retriever.invoke("what is agents")

[Document(id='28aa430d-d77d-48aa-bc16-82f1e742e0be', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, 

In [ ]:
# Initialising the LLM for evaluation

from langchain_groq import ChatGroq
import os

os.environ["GROQ_API_KEY"] = "put your api key here" #for using groq's LLMs with langchain

llm = ChatGroq(
    model="llama-3.3-70b-versatile",  
    temperature=0
)

llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001C291E330B0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001C295FF7250>, model_name='llama-3.3-70b-versatile', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********'))

In [51]:
from langsmith import traceable

#add decorator to target function to trace calls in langsmith
@traceable()
def rag_bot(question: str) -> dict:

    docs = retriever.invoke(question)
    docs_string =" ".join([doc.page_content for doc in docs])

    instructions = f"""You are an helpful assistant who is good at analyzing source info and answering questions based on that info. Use the following retrieved documents to answer the question. If you dont know the answer, say you dont know.  Use three sentences maximum and keep the answer concise.
                        Documents: {docs_string}"""
    
    #invoke llm
    ai_msg = llm.invoke([
        {"role": "system", "content": instructions},
        {"role": "user", "content": question}, 
    ])

    return{"answer": ai_msg.content, "documents":docs}

In [52]:
rag_bot("What is an agent?")

{'answer': 'An agent refers to the speaker of the dialogue who has successfully booked tickets, as indicated by the output "agent" in response to the input "I have successfully booked your tickets". In a broader context, an agent can also refer to an autonomous system powered by a large language model (LLM) that functions as its brain. This type of agent can perform tasks such as problem-solving and learning from mistakes.',
 'documents': [Document(id='5258adb6-e8c2-4aef-86a3-7a16637dc407', metadata={'source': 'https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/', 'title': "Prompt Engineering | Lil'Log", 'description': 'Prompt Engineering, also known as In-Context Prompting, refers to methods for how to communicate with LLM to steer its behavior for desired outcomes without updating the model weights. It is an empirical science and the effect of prompt engineering methods can vary a lot among models, thus requiring heavy experimentation and heuristics.\nThis post only foc

![alt text](<Screenshot 2026-03-03 154845-1.png>)

- Retrieval Relevance: Measures if the Search step successfully found documents that actually relate to the user's Question.

- Groundedness: Checks for hallucinations by ensuring the Answer is derived only from the retrieved Relevant documents.

- Answer Relevance: Evaluates if the final Answer actually addresses the specific Question asked by the user.

- Correctness: Compares the generated Answer against a Reference Answer (the "ground truth") to ensure factual accuracy.

# Creating a new DataSet 

(to be uploaded onto Langsmith for referencing)

In [53]:
from langsmith import Client
client = Client()

# define the examples for dataset

rag_examples  = [
    {
        "inputs": {"question": "How does the ReAct agent use self-reflection? "},
        "outputs": {"answer": "ReAct integrates reasoning and acting, performing actions - such tools like Wikipedia search API - and then observing / reasoning about the tool outputs."},
    },
    {
        "inputs": {"question": "What are the types of biases that can arise with few-shot prompting?"},
        "outputs": {"answer": "The biases that can arise with few-shot prompting include (1) Majority label bias, (2) Recency bias, and (3) Common token bias."},
    },
    {
        "inputs": {"question": "What are five types of adversarial attacks?"},
        "outputs": {"answer": "Five types of adversarial attacks are (1) Token manipulation, (2) Gradient based attack, (3) Jailbreak prompting, (4) Human red-teaming, (5) Model red-teaming."},
    }
]


# creating a dataset and adding examples to LangSmith for evaluation
dataset_name = "RAG Test Evaluation Dataset"
dataset = client.create_dataset(dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=rag_examples
)


{'example_ids': ['622f8340-4a68-4c2a-b9da-e791e5b2bf8e',
  'cbb2af24-c94f-4936-9c0d-b4d7d36d4cec',
  '1e9590f6-550e-4382-a963-b0fab180f092'],
 'count': 3,
 'as_of': '2026-03-03T11:02:30.330373238Z'}

# Evaluators or Metrics

### 1. **Correctness: (Response vs Reference answer)**

- Goal: Measure "how similar/correct is the RAG chain answer, relative to a ground-truth answer"

- Mode: Requires a ground truth (reference) answer supplied through a dataset

- Evaluator: Use LLM-as-judge to assess answer correctness.

In [56]:
from typing_extensions import TypedDict,Annotated

# Correctness Output Schema

class CorrectnessGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your grading decision."]
    correct: Annotated[bool, ..., "True if the answer is correct, False otherwise."]


# correctness prompt for GROQ evaluation
correctness_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and the STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Grade the student answers based ONLY on their factual accuracy relative to the ground truth answer. 
(2) Ensure that the student answer does not contain any conflicting statements.
(3) It is OK if the student answer contains more information than the ground truth answer, as long as it is factually accurate relative to the  ground truth answer.

Correctness:
A correctness value of True means that the student's answer meets all of the criteria.
A correctness value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""



#Initializing GROQ client for evaluation
groq_llm = ChatGroq(
    model="llama-3.3-70b-versatile",  
    temperature=0
)

# Wrapping the GROQ LLM with structured output parsing to ensure it returns a boolean correctness grade along with an explanation for its grading decision.
str_groq_llm = groq_llm.with_structured_output(CorrectnessGrade,
                                                method="json_schema",strict=True)



# evaluator function for correctness using GROQ

def correctness(input: dict, outputs: dict, reference_outputs: dict) -> bool:
    """An evaluator for RAG answer Accuracy"""

    answers =f"""\
    QUESTION: {input['question']}
    GROUND TRUTH ANSWER: {reference_outputs['answer']}
    STUDENT ANSWER: {outputs['answer']}""""Screenshot 2026-03-03 154845-1.png"
    
    #Run evaluator
    grade = str_groq_llm.invoke([
        {"role": "system", "content": correctness_instructions},
        {"role": "user", "content": answers}
    ])

    return grade["correct"]




### 2. **Answer Relevance: (Answer vs Question)**

- Goal: Ensure the "Answer" (Response) directly satisfies the "Question" (Input).

- Mode: Simply looks at the inputs and outputs without needing any reference_outputs.

- Evaluator: Even without a reference answer to grade accuracy, it can still grade relevance—as in, did the model actually address the user's question or not?

In [57]:
# Grade Output Schema

class RelevanceGrade(TypedDict):
    explanation: Annotated[str, ... , "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "Provide the score on whether the answer addresses the question"]

 # Grade Prompt for GROQ evaluation of relevance
relevance_instructions="""You are a teacher grading a quiz. 

You will be given a QUESTION and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION
(2) Ensure the STUDENT ANSWER helps to answer the QUESTION

Relevance:
A relevance value of True means that the student's answer meets all of the criteria.
A relevance value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM for relevance evaluation with structured output parsing to ensure it returns a boolean relevance grade along with an explanation for its grading decision.
relevance_llm = groq_llm.with_structured_output(RelevanceGrade,
                                                method="json_schema",strict=True)

# Evaluator function for relevance using GROQ
def relevance(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer helpfulness."""
    answer = f"QUESTION: {inputs['question']}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = relevance_llm.invoke([
        {"role": "system", "content": relevance_instructions}, 
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]



### 3. **Groundness: (Response vs Retrieved Documents)**

- Goal: Ensure the "Answer" stays strictly within the context of the "Relevant Documents."

- Mode: Provides a way to evaluate responses without needing reference answers by checking if the response is justified by (or "grounded in") the retrieved documents.

- Evaluator: Detects Hallucinations by verifying the AI only uses facts provided in the retrieved text, rather than outside knowledge.

In [58]:
# Grade Output Schema
class GroundedGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    grounded: Annotated[bool, ..., "Provide the score on if the answer hallucinates from the documents"]


# Grade Prompt for GROQ evaluation of groundedness
grounded_instructions = """You are a teacher grading a quiz. 

You will be given FACTS and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is grounded in the FACTS. 
(2) Ensure the STUDENT ANSWER does not contain "hallucinated" information outside the scope of the FACTS.

Grounded:
A grounded value of True means that the student's answer meets all of the criteria.
A grounded value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""


# Grader LLM for groundedness evaluation with structured output parsing to ensure it returns a boolean groundedness grade along with an explanation for its grading decision.
grounded_llm = groq_llm.with_structured_output(GroundedGrade,
                                                method="json_schema",strict=True)
# Evaluator function for groundedness using GROQ
def groundedness(inputs:dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer groundedness."""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = grounded_llm.invoke([{"role": "system", "content": grounded_instructions}, {"role": "user", "content": answer}])
    return grade["grounded"]




### 4. **Retrieval Relevance: (Question vs Documents)**

- Goal: Ensure the "Relevant Documents" actually contain the information needed to answer the "Question."

- Mode: Evaluates the quality of the Search step by checking if retrieved documents relate to the user's query.

- Evaluator: Measures if the search engine avoided "noise" (e.g., returning "cats" when the user asked about "cars").

In [59]:
# Grade output schema
class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "True if the retrieved documents are relevant to the question, False otherwise"]

# Grade prompt
retrieval_relevance_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION and a set of FACTS provided by the student. 

Here is the grade criteria to follow:
(1) You goal is to identify FACTS that are completely unrelated to the QUESTION
(2) If the facts contain ANY keywords or semantic meaning related to the question, consider them relevant
(3) It is OK if the facts have SOME information that is unrelated to the question as long as (2) is met

Relevance:
A relevance value of True means that the FACTS contain ANY keywords or semantic meaning related to the QUESTION and are therefore relevant.
A relevance value of False means that the FACTS are completely unrelated to the QUESTION.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM
retrieval_relevance_llm = groq_llm.with_structured_output(RetrievalRelevanceGrade,
                                                method="json_schema",strict=True)

# Evaluator function for retrieval relevance using GROQ
def retrieval_relevance(inputs: dict, outputs: dict) -> bool:
    """An evaluator for document relevance"""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nQUESTION: {inputs['question']}"

    # Run evaluator
    grade = retrieval_relevance_llm.invoke([
        {"role": "system", "content": retrieval_relevance_instructions}, 
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]

# Run The Final Evaluation

In [ ]:
from langsmith.schemas import Run, Example

# 1. Define your custom evaluator functions
def correctness(run: Run, example: Example) -> dict:
    # Logic to compare model output vs ground truth
    actual = run.outputs.get("output")
    expected = example.outputs.get("answer")
    return {"score": 1 if actual == expected else 0}

def groundedness(run: Run, example: Example) -> dict:
    # Logic to check if response is grounded in retrieved docs
    return {"score": 1} # Placeholder logic

def relevance(run: Run, example: Example) -> dict:
    # Logic to check if response matches input question
    return {"score": 1} # Placeholder logic

def retrieval_relevance(run: Run, example: Example) -> dict:
    # Logic to check if retrieved docs match the question
    return {"score": 1} # Placeholder logic

# 2. Run the experiment with the defined functions
experiment_results = client.evaluate(
    target, 
    data=dataset_name,  
    evaluators=[correctness, groundedness, relevance, retrieval_relevance],
    experiment_prefix="rag-doc-relevance",
    metadata={"model": "llama-3.3-70b-versatile"}
)

# 3. Explore results locally as a dataframe if you have pandas installed
experiment_results.to_pandas()

View the evaluation results for experiment: 'rag-doc-relevance-951c2202' at:
https://smith.langchain.com/o/eaf954a2-4e9e-4932-9036-54070559faa0/datasets/2a48769e-74f3-4b1a-9dfc-d7c1e5fefddb/compare?selectedSessions=d6461df9-cbf1-459d-9620-2e088dcfd7f2




3it [00:03,  1.01s/it]


,inputs.question,outputs.answer,outputs.documents,error,reference.answer,feedback.correctness,feedback.groundedness,feedback.relevance,feedback.retrieval_relevance,execution_time,example_id,id
0,How does the ReAct agent use self-reflection?,The ReAct agent uses self-reflection by comput...,[page_content='Self-reflection is created by s...,None,"ReAct integrates reasoning and acting, perform...",0,1,1,1,1.434464,1e9590f6-550e-4382-a963-b0fab180f092,019cb3d5-1e98-7d03-834c-0ff41abda9dc
1,What are the types of biases that can arise wi...,"According to Zhao et al. (2021), three types o...",[page_content='Zhao et al. (2021) investigated...,None,The biases that can arise with few-shot prompt...,0,1,1,1,0.646719,622f8340-4a68-4c2a-b9da-e791e5b2bf8e,019cb3d5-2441-7991-aa77-1c0c7efb6ba8
2,What are five types of adversarial attacks?,The documents mention that there are various m...,[page_content='The most common way to mitigate...,None,Five types of adversarial attacks are (1) Toke...,0,1,1,1,0.484402,cbb2af24-c94f-4936-9c0d-b4d7d36d4cec,019cb3d5-26cf-73c3-b90e-26d7efe8d8c8


In [68]:
experiment_results

,inputs.question,outputs.answer,outputs.documents,error,reference.answer,feedback.correctness,feedback.groundedness,feedback.relevance,feedback.retrieval_relevance,execution_time,example_id,id
0,How does the ReAct agent use self-reflection?,The ReAct agent uses self-reflection by comput...,[page_content='Self-reflection is created by s...,None,"ReAct integrates reasoning and acting, perform...",0,1,1,1,1.434464,1e9590f6-550e-4382-a963-b0fab180f092,019cb3d5-1e98-7d03-834c-0ff41abda9dc
1,What are the types of biases that can arise wi...,"According to Zhao et al. (2021), three types o...",[page_content='Zhao et al. (2021) investigated...,None,The biases that can arise with few-shot prompt...,0,1,1,1,0.646719,622f8340-4a68-4c2a-b9da-e791e5b2bf8e,019cb3d5-2441-7991-aa77-1c0c7efb6ba8
2,What are five types of adversarial attacks?,The documents mention that there are various m...,[page_content='The most common way to mitigate...,None,Five types of adversarial attacks are (1) Toke...,0,1,1,1,0.484402,cbb2af24-c94f-4936-9c0d-b4d7d36d4cec,019cb3d5-26cf-73c3-b90e-26d7efe8d8c8
